# 1、HumanInTheLoopMiddleware中间件

HumanInTheLoopMiddleware（人在环中间件、人工审核中间件）在工具调用前 中断Agent运行，等待用户对工具调用请求决策。可选的决策有：approve（同意执行）、edit（编辑调用配置后执行）、reject（拒绝执行）。


## 1.1 举例的过程1：工具调用的中断

In [1]:

from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os


# 1、提供大模型
load_dotenv(override=True)

model = init_chat_model(
    model="qwen3.7-plus",
    model_provider="openai",
    profile={"max_input_tokens": 128_000},
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    # temperature=1.5,
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
    extra_body={"enable_thinking": False},
    # max_tokens=10,
)

model_with_zhipuai = init_chat_model(
    model="GLM-4.5-Air",
    model_provider="openai",
    api_key=os.getenv("ZHIPUAI_API_KEY"),
    base_url=os.getenv("ZHIPUAI_BASE_URL"),
    extra_body={"enable_thinking": False}
)

In [13]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage
from langchain.tools import tool
from langgraph.types import Command
from rich import print as rprint

@tool
def get_weather(city: str, is_forcast: bool = False) -> str:
    """
    查询指定城市天气

    Args:
        city: 城市名称
        is_forcast: 是否包含明日天气预报？
    """
    res = f"{city}今天天气不错"
    if is_forcast:
        res += "\n明天下雨"
    return res

@tool
def get_news() -> str:
    """
    查询当日新闻
    """
    return "中方三艘油轮通过霍尔木兹海峡"

@tool
def read_email_tool(email_id: str) -> str:
    """通过邮件ID读取内容的伪函数"""
    return f"邮件ID：{email_id}\n是空的"

@tool
def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """发送邮件伪函数"""
    print(">>> 真的执行发送邮件工具了")
    return f"发送给 {recipient} 的邮件标题是：{subject}，内容：{body}"

agent = create_agent(
    model=model_with_zhipuai,
    tools=[get_weather, get_news, read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "get_weather": True,
                "get_news": True,
                "read_email_tool": False,
                "send_email_tool": {
                    "allowed_decisions" : ["approve", "reject"],
                    "description" : "发送邮件中断了..."
                },
            },
            description_prefix="中断啦！！"
        )
    ],
)
config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {
    "messages" : [HumanMessage(content="请帮我查询今天北京的天气"
                               "查询今日新闻"
                               "查看ID为 'sk2131421' 的邮件内容，"
                               "向15641685664@qq.com发送邮件，标题是'哈哈哈'，内容是：'你好啊'"
                               "同时做这四件事")]
    },
    config=config
)

rprint(response)

{
    'messages': [
        HumanMessage(
            content="请帮我查询今天北京的天气查询今日新闻查看ID为 'sk2131421' 
的邮件内容，向15641685664@qq.com发送邮件，标题是'哈哈哈'，内容是：'你好啊'同时做这四件事",
            additional_kwargs={},
            response_metadata={},
            id='c55d9d68-ef1b-4764-810e-c385b8bb2a4a'
        ),
        AIMessage(
            content='\n我来帮您同时完成这四件事：\n',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 273,
                    'prompt_tokens': 408,
                    'total_tokens': 681,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 407}
                },
                'model_provider': 'openai',
                'model_name': 'GLM-4.5-Air',
                'system_fingerprint': None,
                'id': '2026071817213949ed3784e4d24ddf',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019f7488-094b-73d0-96ea-50a213632a56-0',
            tool_calls=[
                {
                    'name': 'get_weather',
                    'args': {'city': '北京', 'is_forcast': False},
                    'id': 'call_-7437639740301502086',
                    'type': 'tool_call'
                },
                {
                    'name': 'read_email_tool',
                    'args': {'email_id': 'sk2131421'},
                    'id': 'call_-7437639740301502085',
                    'type': 'tool_call'
                },
                {
                    'name': 'send_email_tool',
                    'args': {'recipient': '15641685664@qq.com', 'subject': '哈哈哈', 'body': '你好啊'},
                    'id': 'call_-7437639740301502084',
                    'type': 'tool_call'
                },
                {'name': 'get_news', 'args': {}, 'id': 'call_-7437639740301502083', 'type': 'tool_call'}
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 408,
                'output_tokens': 273,
                'total_tokens': 681,
                'input_token_details': {'cache_read': 407},
                'output_token_details': {}
            }
        )
    ],
    '__interrupt__': [
        Interrupt(
            value={
                'action_requests': [
                    {
                        'name': 'get_weather',
                        'args': {'city': '北京', 'is_forcast': False},
                        'description': "中断啦！！\n\nTool: get_weather\nArgs: {'city': '北京', 'is_forcast': 
False}"
                    },
                    {
                        'name': 'send_email_tool',
                        'args': {'recipient': '15641685664@qq.com', 'subject': '哈哈哈', 'body': '你好啊'},
                        'description': '发送邮件中断了...'
                    },
                    {'name': 'get_news', 'args': {}, 'description': '中断啦！！\n\nTool: get_news\nArgs: {}'}
                ],
                'review_configs': [
                    {'action_name': 'get_weather', 'allowed_decisions': ['approve', 'edit', 'reject']},
                    {'action_name': 'send_email_tool', 'allowed_decisions': ['approve', 'reject']},
                    {'action_name': 'get_news', 'allowed_decisions': ['approve', 'edit', 'reject']}
                ]
            },
            id='458125e457b221f2c553082572393629'
        )
    ]
}

## 举例的过程2：指明工具调用请求决策

In [14]:


weather_decision = {
    "type" : "edit",
    "edited_action" : {
        "name" : "get_weather",
        "args" : {"city" : "上海市", "is_forcast" : True},
    }
}

news_decision = {
    "type" : "approve"
}

send_email_decision = {
    "type" : "approve"
}



decisions = {
    "decisions" : []
}

interrupts = response.get("__interrupt__", [])
action_requests = interrupts[0].value["action_requests"]


for action in action_requests:
    if action["name"] == "get_weather":
        decisions["decisions"].append(weather_decision)
    if action["name"] == "get_news":
        decisions["decisions"].append(news_decision)
    if action["name"] == "send_email_tool":
        decisions["decisions"].append(send_email_decision)

if interrupts:
    resumed_response = agent.invoke(
        Command(resume=decisions),
        config = config

    )

    for msg in resumed_response["messages"]:
        msg.pretty_print()


>>> 真的执行发送邮件工具了
================================ Human Message =================================

请帮我查询今天北京的天气查询今日新闻查看ID为 'sk2131421' 的邮件内容，向15641685664@qq.com发送邮件，标题是'哈哈哈'，内容是：'你好啊'同时做这四件事
================================== Ai Message ==================================


我来帮您同时完成这四件事：
Tool Calls:
  get_weather (call_-7437639740301502086)
 Call ID: call_-7437639740301502086
  Args:
    city: 上海市
    is_forcast: True
  read_email_tool (call_-7437639740301502085)
 Call ID: call_-7437639740301502085
  Args:
    email_id: sk2131421
  send_email_tool (call_-7437639740301502084)
 Call ID: call_-7437639740301502084
  Args:
    recipient: 15641685664@qq.com
    subject: 哈哈哈
    body: 你好啊
  get_news (call_-7437639740301502083)
 Call ID: call_-7437639740301502083
  Args:
================================= Tool Message =================================
Name: get_weather

上海市今天天气不错
明天下雨
================================= Tool Message =================================
Name: read_email_tool

邮件ID：sk21